# Traditional NER vs. LLM Parsing — Experiment Walkthrough

**Research question:** How does task-specific entity recognition compare with prompt-based structured extraction? We use **50 longer English test passages**, **10 Turkish supplementary passages**, and **7 separate development passages**.

**This notebook is runnable on your Mac.** First follow the installation steps in `README.md`. For full, repeated experiments use `scripts/run_experiments.py`; the notebook demonstrates the logic step by step. **Do not include development examples in final test F1.**

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "english_test.jsonl").exists():
    ROOT = ROOT.parent  # if opened in notebooks/ directory
sys.path.insert(0, str(ROOT))
from src.dataset import load_data, gold_for, CORE_LABELS
from src.evaluation import score_entities, summarize
from src.models import run_spacy, run_ollama
english = load_data('english_test.jsonl')
turkish = load_data('turkish_supplementary.jsonl')
development = load_data('development.jsonl')
print(f"EN test: {len(english)} | TR supplementary: {len(turkish)} | DEV: {len(development)}")
print('First English text length (words):', len(english[0]['text'].split()))

## 1. Inspect a longer paragraph and its gold spans

`PERSON`, `ORG`, `LOC` are **core** categories. `DATE`, `PRODUCT`, and `JOB_TITLE` are recorded in a separate field for auxiliary experiments. All spans are exact character offsets in the original text.

In [ ]:
sample = english[30]  # EN031: job title + organization + city + year
print('Sample:', sample['id'], '\n')
print(sample['text'])
pd.DataFrame(sorted(sample['entities'] + sample['extended_entities'], key=lambda e:e['start']))

## 2. Run Traditional NER (spaCy)

Install `en_core_web_sm` once with `python -m spacy download en_core_web_sm` in Terminal. spaCy predicts entities with predefined labels. We normalize `GPE` and `LOC` to a common `LOC` for the main comparison.

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')
print('Available NER labels:', nlp.get_pipe('ner').labels)
ner_result = run_spacy(sample['text'], nlp=nlp)
print('spaCy prediction:')
print(pd.DataFrame(ner_result['entities']).to_string(index=False))
print('spaCy core F1 for this single sample:', score_entities(gold_for(sample), ner_result['entities'])['f1'])

## 3. Run a local Ollama LLM

Run `ollama pull qwen2.5:3b` once and make sure Ollama is running. The adapter connects **only** to `http://localhost:11434`. We ask for entity strings that appear exactly in the source and use JSON Schema to constrain their shape.

In [ ]:
llm_result = run_ollama(sample['text'], model='qwen2.5:3b', format_mode='schema')
print('Raw LLM response:')
print(llm_result['raw_text'])
print('JSON valid:', llm_result['raw_json_valid'], '| schema valid:', llm_result['schema_valid'])
print('LLM predicted spans:')
print(pd.DataFrame(llm_result['entities']).to_string(index=False))
print('Ungrounded / unsupported outputs:', llm_result['ungrounded'])
print('LLM core F1 for this single sample:', score_entities(gold_for(sample), llm_result['entities'])['f1'])

## 4. New-category demonstration (`JOB_TITLE`)

The spaCy checkpoint's built-in NER label set has no `JOB_TITLE`. That does **not** prove NER can never learn it; a new model head / fine-tuning or rule augmentation can add the category. We therefore demonstrate **out-of-the-box adaptability**, not a head-to-head F1 where unsupported NER is unfairly scored as zero.

In [ ]:
print('spaCy supports JOB_TITLE:', 'JOB_TITLE' in nlp.get_pipe('ner').labels)
print('Gold job title:', gold_for(sample, ['JOB_TITLE']))
job_result = run_ollama(sample['text'], labels=('JOB_TITLE',), model='qwen2.5:3b')
print('LLM JOB_TITLE output:', job_result['entities'])
print('LLM JOB_TITLE score:', score_entities(gold_for(sample,['JOB_TITLE']), job_result['entities']))

## 5. Turkish supplementary analysis

The Turkish passages are longer and contain inflected forms such as `Google'a`. For this study we **do not** treat English `en_core_web_sm` as a Turkish NER baseline. Compare LLM parsing and the character boundaries instead.

In [ ]:
turkish_example = turkish[0]
print(turkish_example['text'])
print(pd.DataFrame(turkish_example['entities']).to_string(index=False))
print('Review apostrophe suffixes in these spans before scoring.')

## 6. Run controlled experiments from the project root

Each command writes a detailed `CSV` and a summary `JSON` to `results/`. Default `--limit 5` is a safe first run. `--limit 0` runs the complete relevant test set. Run the commands in Terminal, then use the next cell to inspect results.

Suggested sequence:

```bash
python scripts/run_experiments.py --experiment accuracy --limit 5
python scripts/run_experiments.py --experiment json --limit 5
python scripts/run_experiments.py --experiment flexibility --limit 5
python scripts/run_experiments.py --experiment ambiguity --limit 5
python scripts/run_experiments.py --experiment language --limit 2
```

Then expand to full data after checking output quality. Read `README.md` for all 10 experiment codes.

In [ ]:
result_file = ROOT / 'results' / 'accuracy_detail.csv'
if result_file.exists():
    df = pd.read_csv(result_file)
    display(df[['sample_id','method','tp','fp','fn','precision','recall','f1','latency_sec']].head(15))
else:
    print('No measured results yet — run the accuracy command in Terminal first.')

## Interpretation checklist

- `F1` is computed from **exact character spans and entity types**, with micro-averaging for the whole set.
- Keep **JSON validity separate from entity correctness**.
- An LLM entity that cannot be anchored to the source is reported and counted as an additional false positive.
- Compare only categories that both systems support. **JOB_TITLE is an adaptability demonstration**.
- Review errors manually and report the limitations of synthetic passages, small samples, and different language-specific checkpoints.
- No benchmark scores have been pre-generated in this project; any numbers must come from your actual model runs.